Purpose
---
This script generates descriptive statistics for a Wikibase-based knowledge graph
through SPARQL queries. It computes distributions of items by class and category,
counts statements and qualifiers by property, and analyzes referenced statements
grouped by source provenance, exporting all results as CSV reports.

# Setup

In [1]:
import requests
import csv
import time

ENDPOINT = "effkg"
SOURCE_PROP = "P46"
SPARQL_ENDPOINT = f"https://{ENDPOINT}.wikibase.cloud/query/sparql"

HEADERS = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "statement-counter/1.0"
}

"""Execute a SPARQL query against the configured endpoint."""
def run_query(query):
    r = requests.get(
        SPARQL_ENDPOINT,
        params={"query": query, "format": "json"},
        headers=HEADERS,
        timeout=120
    )
    r.raise_for_status()
    return r.json()

# Items by type

In [2]:
INSTANCE_OF = "P35"
SUBCLASS_OF = "P52"

# -------------------------------------------------------------------
# Retrieve all entities that are subclass of another entity
# -------------------------------------------------------------------

classes_query = f"""
PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?classId ?classLabel ?parentClassId ?parentClassLabel
WHERE {{
  ?class wdt:{SUBCLASS_OF} ?parentClass .

  OPTIONAL {{
    ?class rdfs:label ?classLabel .
    FILTER(LANG(?classLabel) = "en")
  }}

  OPTIONAL {{
    ?parentClass rdfs:label ?parentClassLabel .
    FILTER(LANG(?parentClassLabel) = "en")
  }}

  BIND(REPLACE(STR(?class), "^.*/entity/", "") AS ?classId)
  BIND(REPLACE(STR(?parentClass), "^.*/entity/", "") AS ?parentClassId)
}}
ORDER BY ?classId
"""

class_results = run_query(classes_query)

classes = []

for b in class_results["results"]["bindings"]:
    class_id = b["classId"]["value"]
    parent_class_id = b["parentClassId"]["value"]

    class_label = b.get("classLabel", {}).get("value", class_id)
    parent_class_label = b.get("parentClassLabel", {}).get("value", parent_class_id)

    classes.append(
        (
            class_id,
            class_label,
            parent_class_id,
            parent_class_label,
        )
    )

print(f"Classes found: {len(classes):,}")

# -------------------------------------------------------------------
# Count instances for each class
# -------------------------------------------------------------------

rows = []
total_instances = 0

for class_id, class_label, parent_class_id, parent_class_label in classes:
    q = f"""
    PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
    PREFIX wd: <https://{ENDPOINT}.wikibase.cloud/entity/>

    SELECT (COUNT(DISTINCT ?item) AS ?count)
    WHERE {{
      ?item wdt:{INSTANCE_OF} wd:{class_id} .
    }}
    """

    try:
        res = run_query(q)
        count = int(res["results"]["bindings"][0]["count"]["value"])

        if count > 0:
            rows.append(
                (
                    class_id,
                    class_label,
                    parent_class_id,
                    parent_class_label,
                    count,
                )
            )

            total_instances += count

    except Exception as e:
        print(f"Error with {class_id} ({class_label}): {e}")

# -------------------------------------------------------------------
# Sort descending by number of instances
# -------------------------------------------------------------------

rows.sort(key=lambda x: x[4], reverse=True)

# -------------------------------------------------------------------
# Pretty-print results table
# -------------------------------------------------------------------

def print_table(rows):
    headers = [
        "Class",
        "Class label",
        "Parent",
        "Parent label",
        "Instances"
    ]

    table_rows = [
        [
            class_id,
            class_label,
            parent_class_id,
            parent_class_label,
            f"{count:,}",
        ]
        for class_id, class_label, parent_class_id, parent_class_label, count in rows
    ]

    all_rows = [headers] + table_rows

    col_widths = [
        max(len(str(row[i])) for row in all_rows)
        for i in range(len(headers))
    ]

    separator = "-+-".join("-" * width for width in col_widths)

    print("\n=== ITEMS BY CLASS ===\n")

    print(
        " | ".join(
            headers[i].ljust(col_widths[i])
            for i in range(len(headers))
        )
    )

    print(separator)

    for row in table_rows:
        print(
            " | ".join(
                str(row[i]).ljust(col_widths[i])
                for i in range(len(headers))
            )
        )

# -------------------------------------------------------------------
# Display results
# -------------------------------------------------------------------

print_table(rows)

print(f"\nTOTAL INSTANCE RELATIONS: {total_instances:,}")

# -------------------------------------------------------------------
# Export results to CSV
# -------------------------------------------------------------------

with open("items_by_class.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")

    writer.writerow(
        [
            "Class",
            "ClassLabel",
            "ParentClass",
            "ParentClassLabel",
            "NumberOfInstances",
        ]
    )

    writer.writerows(rows)

    writer.writerow([])
    writer.writerow(["TOTAL", "", "", "", total_instances])

print("\nCSV exported: items_by_class.csv")

Classes found: 13

=== ITEMS BY CLASS ===

Class  | Class label                                  | Parent | Parent label                 | Instances
-------+----------------------------------------------+--------+------------------------------+----------
Q24    | Fishing Vessel                               | Q23    | Vessel                       | 185,025  
Q398   | Vessel supporting fishing related activities | Q23    | Vessel                       | 3,145    
Q62    | Fishing Port                                 | Q488   | Port                         | 1,997    
Q13    | Fishing Area                                 | Q28608 | Maritime administrative unit | 114      
Q138   | Maritimal District                           | Q28608 | Maritime administrative unit | 108      
Q63    | Maritimal Province                           | Q28608 | Maritime administrative unit | 30       
Q28610 | Spanish Autonomous Community                 | Q379   | Land administrative unit     | 17       
Q47

## Categories

In [3]:
INSTANCE_OF = "P35"
CATEGORY_OF = "P50"
CATEGORY_QID = "Q473"

ITEM_CATEGORY_PROPERTIES = [
    INSTANCE_OF,
    "P25",
]

query = f"""
PREFIX wdt: <https://{ENDPOINT}.wikibase.cloud/prop/direct/>
PREFIX wd: <https://{ENDPOINT}.wikibase.cloud/entity/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?categoryId ?categoryLabel
       ?categoryOfId ?categoryOfLabel
       ?subCategoryId ?subCategoryLabel
       (COUNT(DISTINCT ?item) AS ?numberOfItems)
WHERE {{
  ?category wdt:{INSTANCE_OF} wd:{CATEGORY_QID} .

  OPTIONAL {{
    ?category wdt:{CATEGORY_OF} ?categoryOf .
    BIND(REPLACE(STR(?categoryOf), "^.*/entity/", "") AS ?categoryOfId)

    OPTIONAL {{
      ?categoryOf rdfs:label ?categoryOfLabel .
      FILTER(LANG(?categoryOfLabel) = "en")
    }}
  }}

  ?subCategory wdt:{INSTANCE_OF} ?category .

  VALUES ?itemCategoryProperty {{
    {" ".join("wdt:" + p for p in ITEM_CATEGORY_PROPERTIES)}
  }}

  OPTIONAL {{
    ?item ?itemCategoryProperty ?subCategory .
  }}

  BIND(REPLACE(STR(?category), "^.*/entity/", "") AS ?categoryId)
  BIND(REPLACE(STR(?subCategory), "^.*/entity/", "") AS ?subCategoryId)

  OPTIONAL {{
    ?category rdfs:label ?categoryLabel .
    FILTER(LANG(?categoryLabel) = "en")
  }}

  OPTIONAL {{
    ?subCategory rdfs:label ?subCategoryLabel .
    FILTER(LANG(?subCategoryLabel) = "en")
  }}
}}
GROUP BY ?categoryId ?categoryLabel ?categoryOfId ?categoryOfLabel ?subCategoryId ?subCategoryLabel
ORDER BY ?categoryLabel DESC(?numberOfItems)
"""

results = run_query(query)

rows = []

for b in results["results"]["bindings"]:
    category_id = b["categoryId"]["value"]
    category_label = b.get("categoryLabel", {}).get("value", category_id)

    category_of_id = b.get("categoryOfId", {}).get("value", "")
    category_of_label = b.get("categoryOfLabel", {}).get("value", category_of_id)

    subcategory_id = b["subCategoryId"]["value"]
    subcategory_label = b.get("subCategoryLabel", {}).get("value", subcategory_id)

    count = int(b["numberOfItems"]["value"])

    rows.append(
        (
            category_id,
            category_label,
            category_of_id,
            category_of_label,
            subcategory_id,
            subcategory_label,
            count,
        )
    )


def print_table(rows):
    headers = [
        "Category",
        "Category label",
        "Category of",
        "Category of label",
        "Subcategory",
        "Subcategory label",
        "Items",
    ]

    table_rows = [
        [
            category_id,
            category_label,
            category_of_id,
            category_of_label,
            subcategory_id,
            subcategory_label,
            f"{count:,}",
        ]
        for (
            category_id,
            category_label,
            category_of_id,
            category_of_label,
            subcategory_id,
            subcategory_label,
            count,
        ) in rows
    ]

    all_rows = [headers] + table_rows

    col_widths = [
        max(len(str(row[i])) for row in all_rows)
        for i in range(len(headers))
    ]

    separator = "-+-".join("-" * width for width in col_widths)

    print("\n=== CATEGORY ITEMS: SECOND LEVEL ===\n")
    print(" | ".join(headers[i].ljust(col_widths[i]) for i in range(len(headers))))
    print(separator)

    for row in table_rows:
        print(" | ".join(str(row[i]).ljust(col_widths[i]) for i in range(len(headers))))


print_table(rows)

total_subcategories = len(rows)
total_items = sum(row[6] for row in rows)

print(f"\nTOTAL SUBCATEGORIES: {total_subcategories:,}")
print(f"TOTAL ITEM RELATIONS: {total_items:,}")

with open("category_items_second_level.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")

    writer.writerow(
        [
            "Category",
            "CategoryLabel",
            "CategoryOf",
            "CategoryOfLabel",
            "Subcategory",
            "SubcategoryLabel",
            "NumberOfItems",
        ]
    )

    writer.writerows(rows)

    writer.writerow([])
    writer.writerow(["TOTAL SUBCATEGORIES", total_subcategories, "", "", "", "", ""])
    writer.writerow(["TOTAL ITEM RELATIONS", "", "", "", "", "", total_items])

print("\nCSV exported: category_items_second_level.csv")


=== CATEGORY ITEMS: SECOND LEVEL ===

Category | Category label        | Category of | Category of label | Subcategory | Subcategory label                                              | Items  
---------+-----------------------+-------------+-------------------+-------------+----------------------------------------------------------------+--------
Q472     | Fishing Gear Category | Q469        | Fishing Gear      | Q511        | Set gillnets (anchored)                                        | 106,012
Q472     | Fishing Gear Category | Q469        | Fishing Gear      | Q529        | Set longlines                                                  | 60,998 
Q472     | Fishing Gear Category | Q469        | Fishing Gear      | Q546        | Gear not known                                                 | 51,925 
Q472     | Fishing Gear Category | Q469        | Fishing Gear      | Q515        | Trammel nets                                                   | 39,274 
Q472     | Fishing Gear C

# Statements by property

In [4]:
# Get all property IDs and labels
properties_query = """
PREFIX wikibase: <http://wikiba.se/ontology#>

SELECT ?propId ?propertyLabel
WHERE {
  ?property a wikibase:Property ;
            wikibase:claim ?p ;
            rdfs:label ?propertyLabel .
  FILTER(LANG(?propertyLabel) = "en")
  BIND(REPLACE(STR(?property), "^.*/entity/", "") AS ?propId)
}
ORDER BY ?propId
"""

prop_results = run_query(properties_query)
properties = [
    (b["propId"]["value"], b["propertyLabel"]["value"])
    for b in prop_results["results"]["bindings"]
]

rows = []
total = 0

for prop_id, prop_label in properties:
    q = f"""
    PREFIX p: <https://{ENDPOINT}.wikibase.cloud/prop/>

    SELECT (COUNT(?statement) AS ?count)
    WHERE {{
      ?item p:{prop_id} ?statement .
    }}
    """
    try:
        res = run_query(q)
        count = int(res["results"]["bindings"][0]["count"]["value"])
        if count > 0:
            rows.append((prop_id, prop_label, count))
            total += count
    except Exception as e:
        print(f"Error with {prop_id} ({prop_label}): {e}")

rows.sort(key=lambda x: x[2], reverse=True)

print("=== STATEMENTS BY PROPERTY ===")
for prop_id, prop_label, count in rows:
    print(f"{prop_id}\t{prop_label}\t{count}")

print(f"\nTOTAL STATEMENTS: {total}")

with open("statements_by_property.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow(["Property", "Label", "NumberOfStatements"])
    writer.writerows(rows)
    writer.writerow([])
    writer.writerow(["TOTAL", "", total])

=== STATEMENTS BY PROPERTY ===
P25	has fishing gear category	371013
P23	main engine power	213317
P35	instance of	190592
P12	CFR	188177
P51	has vessel category	187575
P55	country of registration	185685
P9	external marking	185617
P59	segment	184804
P10	registration number	183401
P29	entry into service date	183396
P37	name	182619
P16	LOA	181359
P27	hull material	175595
P30	date of construction	160307
P20	GT Tonnage	153629
P21	other tonnage	126242
P11	registration place	124760
P28	on board equipment	115614
P17	LBP	77697
P58	date of retirement	71896
P57	date of destruction	42551
P26	code	32992
P32	status on national registry	30730
P49	IMO	30529
P31	registration on national registry	30527
P56	administration responsible of registration	30527
P41	operates at	23829
P15	MMSI	20309
P24	auxiliar engine power	9499
P45	number of vessels	5177
P14	UVI	3868
P42	coordinates	1927
P3	belongs to administrative unit	430
P22	safety tonnage GTs	338
P7	has ports	295
P60	administrated by	155
P43	contains admini

# Qualifiers by property

In [5]:
# Retrieve properties that can act as qualifiers
qualifier_props_query = """
PREFIX wikibase: <http://wikiba.se/ontology#>

SELECT ?propId ?propertyLabel
WHERE {
  ?property a wikibase:Property ;
            wikibase:qualifier ?pqProp ;
            rdfs:label ?propertyLabel .
  FILTER(LANG(?propertyLabel) = "en")
  BIND(REPLACE(STR(?property), "^.*/entity/", "") AS ?propId)
}
ORDER BY ?propId
"""

results = run_query(qualifier_props_query)
props = [
    (b["propId"]["value"], b["propertyLabel"]["value"])
    for b in results["results"]["bindings"]
]

rows = []
total = 0

for prop_id, prop_label in props:
    q = f"""
    PREFIX pq: <https://{ENDPOINT}.wikibase.cloud/prop/qualifier/>

    SELECT (COUNT(DISTINCT ?statement) AS ?count)
    WHERE {{
      ?statement pq:{prop_id} ?value .
    }}
    """
    try:
        res = run_query(q)
        count = int(res["results"]["bindings"][0]["count"]["value"])
        if count > 0:
            rows.append((prop_id, prop_label, count))
            total += count
            print(f"{prop_id}\t{prop_label}\t{count}")
    except Exception as e:
        print(f"Error with {prop_id} ({prop_label}): {e}")

rows.sort(key=lambda x: x[2], reverse=True)

print("\n=== QUALIFIERS BY PROPERTY ===")
for prop_id, prop_label, count in rows:
    print(f"{prop_id}\t{prop_label}\t{count}")

print(f"\nTOTAL QUALIFIED STATEMENTS COUNTED: {total}")

P13	IRCS	52858
P32	status on national registry	3
P33	since	30730
P44	date	5180
P47	start date	129
P48	end date	57
P55	country of registration	12682

=== QUALIFIERS BY PROPERTY ===
P13	IRCS	52858
P33	since	30730
P55	country of registration	12682
P44	date	5180
P47	start date	129
P48	end date	57
P32	status on national registry	3

TOTAL QUALIFIED STATEMENTS COUNTED: 101639


# References by source value

In [6]:
SOURCES = [
    "https://webgate.ec.europa.eu/fleet-europa",
    "https://servicio.pesca.mapama.es/CENSO/ConsultaBuqueRegistro/Buques/Search",
    "https://signa.ign.es/signa/",
    "https://boe.es/buscar/pdf/2007/BOE-A-2007-10951-consolidado.pdf",
    "https://www.mapa.gob.es/es/pesca/temas/registro-flota/informacion-sobre-flota-pesquera",
    "https://openknowledge.fao.org/handle/20.500.14283/cb5201en",
    "https://openknowledge.fao.org/items/4de1cb7a-34e3-4645-ae94-0cfc3ef4ad70",
    "https://openknowledge.fao.org/server/api/core/bitstreams/acee638a-d119-4fc6-8512-d7d953b14cd8/content",
    "https://app.hubocean.earth/catalog",
    "https://msi.nga.mil",
    "https://www.data.gouv.fr/api/1/datasets/r/60fe965d-5888-493b-9321-24bc3b1f84db",
    "https://www.fao.org/fishery/en/area",
    "https://www.boe.es/eli/es/rd/2007/05/18/638/con",
]

"""Execute a SPARQL query with retry and exponential backoff support."""
def run_query_timeout(query, timeout=120, max_retries=4, backoff_base=2):
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                SPARQL_ENDPOINT,
                params={"query": query, "format": "json"},
                headers=HEADERS,
                timeout=timeout
            )

            if response.status_code >= 500:
                raise requests.HTTPError(
                    f"HTTP {response.status_code}: {response.text[:300]}",
                    response=response
                )

            response.raise_for_status()
            return response.json()

        except requests.RequestException as e:
            last_error = e
            if attempt == max_retries:
                break

            sleep_seconds = backoff_base ** (attempt - 1)
            print(f"  attempt {attempt}/{max_retries} failed: {e}")
            print(f"  retrying in {sleep_seconds}s...")
            time.sleep(sleep_seconds)

    raise last_error

"""Count statements referencing a specific source URL."""
def count_statements_for_source(source_url):
    query = f"""
    PREFIX prov: <http://www.w3.org/ns/prov#>
    PREFIX pr: <https://{ENDPOINT}.wikibase.cloud/prop/reference/>

    SELECT (COUNT(DISTINCT ?statement) AS ?count)
    WHERE {{
      ?statement prov:wasDerivedFrom ?reference .
      ?reference pr:{SOURCE_PROP} <{source_url}> .
    }}
    """

    result = run_query_timeout(query)
    bindings = result.get("results", {}).get("bindings", [])
    if not bindings:
        return 0

    return int(bindings[0]["count"]["value"])

"""Generate statistics for statements grouped by reference source."""
def main():
    rows = []
    total = 0

    print("=== REFERENCES BY SOURCE VALUE ===")

    for idx, source in enumerate(SOURCES, start=1):
        print(f"[{idx}/{len(SOURCES)}] {source}")

        try:
            count = count_statements_for_source(source)
            rows.append({
                "source": source,
                "numberOfStatements": count,
                "status": "OK",
                "error": ""
            })
            total += count
            print(f"  -> {count}")

        except Exception as e:
            rows.append({
                "source": source,
                "numberOfStatements": "",
                "status": "ERROR",
                "error": str(e)
            })
            print(f"  -> ERROR: {e}")

        time.sleep(1)

    print(f"\nTOTAL REFERENCED STATEMENTS: {total}")

    rows_sorted = sorted(
        rows,
        key=lambda r: r["numberOfStatements"] if isinstance(r["numberOfStatements"], int) else -1,
        reverse=True
    )

    with open("references_by_source.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow(["source", "numberOfStatements", "status", "error"])

        for row in rows_sorted:
            writer.writerow([
                row["source"],
                row["numberOfStatements"],
                row["status"],
                row["error"]
            ])

        writer.writerow([])
        writer.writerow(["TOTAL", total, "", ""])

    print("\nCSV generated: references_by_source.csv")

if __name__ == "__main__":
    main()

=== REFERENCES BY SOURCE VALUE ===
[1/13] https://webgate.ec.europa.eu/fleet-europa
  -> 3448294
[2/13] https://servicio.pesca.mapama.es/CENSO/ConsultaBuqueRegistro/Buques/Search
  -> 570603
[3/13] https://signa.ign.es/signa/
  -> 193
[4/13] https://boe.es/buscar/pdf/2007/BOE-A-2007-10951-consolidado.pdf
  -> 293
[5/13] https://www.mapa.gob.es/es/pesca/temas/registro-flota/informacion-sobre-flota-pesquera
  -> 5158
[6/13] https://openknowledge.fao.org/handle/20.500.14283/cb5201en
  -> 189
[7/13] https://openknowledge.fao.org/items/4de1cb7a-34e3-4645-ae94-0cfc3ef4ad70
  -> 367
[8/13] https://openknowledge.fao.org/server/api/core/bitstreams/acee638a-d119-4fc6-8512-d7d953b14cd8/content
  -> 9
[9/13] https://app.hubocean.earth/catalog
  -> 4
[10/13] https://msi.nga.mil
  -> 995
[11/13] https://www.data.gouv.fr/api/1/datasets/r/60fe965d-5888-493b-9321-24bc3b1f84db
  -> 6313
[12/13] https://www.fao.org/fishery/en/area
  -> 569
[13/13] https://www.boe.es/eli/es/rd/2007/05/18/638/con
  -> 858
